# Milestone 1: Exploratory Data Analysis & Dashboard Prototype
## Interactive Online Sales Performance & Customer Behavior Dashboard
**Course:** Data Visualization | **Date:** April 2026

---

## 1. Introduction & Industry Motivation

### 1.1 Why This Dashboard Matters in Industry

The global e-commerce market exceeded **\$5.8 trillion in retail sales in 2023** and is projected to surpass \$8 trillion by 2027 (Statista, 2024). As online retail scales at this pace, organizations face a critical challenge: **they collect enormous volumes of transactional data but lack the tools to interpret it quickly enough to make timely business decisions.**

In industry, sales and marketing managers routinely ask questions such as:
- *Which product categories are driving revenue this quarter?*
- *Which countries represent our highest-value markets?*
- *Are we retaining our best customers, or losing them to competitors?*
- *At what time of day or day of week do orders peak?*

Without a visual analytics layer, answering these questions requires a data analyst, SQL queries, and spreadsheet exports — a process that can take **days**. A well-designed interactive dashboard collapses this to **seconds**, empowering non-technical stakeholders (store managers, product leads, marketing directors) to explore the data themselves.

### 1.2 Research Evidence Supporting This Need

The need for visual, interactive business dashboards is well established in academic literature:

**[1] Sarikaya et al. (2019)** conducted a systematic study of how dashboards are used in practice across industry and found that the primary value of dashboards is enabling *rapid awareness* and *situation monitoring* for decision-makers who do not have data engineering backgrounds. They found that effective dashboards reduce decision latency significantly by surfacing aggregated KPIs and interactive drill-downs in a single interface. This directly motivates our multi-KPI layout with sidebar filters.
> *Sarikaya, A., Correll, M., Bartram, L., Tory, M., & Fisher, D. (2019). What do we talk about when we talk about dashboards? IEEE Transactions on Visualization and Computer Graphics, 25(1), 682–692.*

**[2] Heer & Shneiderman (2012)** established that *interactive dynamics* — the ability to filter, zoom, and query data visually — dramatically improve analytical insight compared to static reports. Their framework describes how features like brushing, filtering, and details-on-demand (hover tooltips) reduce cognitive load and help users identify patterns they would not find in raw tables. Our dashboard implements all three of these interaction types.
> *Heer, J., & Shneiderman, B. (2012). Interactive dynamics for visual analysis. Queue, 10(2), 30–55.*

**[3] Ngai, Xiu & Chau (2009)** reviewed 900+ papers on data mining in customer relationship management (CRM) and found that **customer segmentation and purchase behavior analysis** are the most impactful applications of analytics in retail. They specifically highlight RFM (Recency, Frequency, Monetary) analysis as the dominant technique used by industry for customer retention decisions — a technique our dashboard implements directly.
> *Ngai, E.W.T., Xiu, L., & Chau, D.C.K. (2009). Application of data mining techniques in customer relationship management: A literature review and classification. Expert Systems with Applications, 36(2), 2592–2602.*

**[4] Chen & Popovich (2003)** argued that understanding customer behavior through transaction data is not merely a technical exercise but a **strategic business imperative**. Organizations that integrate CRM analytics into their operational workflow see measurable improvements in customer retention rates, which matters because acquiring a new customer costs 5–7× more than retaining an existing one.
> *Chen, I.J., & Popovich, K. (2003). Understanding customer relationship management (CRM): People, process and technology. Business Process Management Journal, 9(5), 672–688.*

### 1.3 Who Will Use This Dashboard and How

| Stakeholder | Use Case | Key Charts |
|---|---|---|
| **Sales Manager** | Track monthly revenue trends, identify slow periods | Revenue timeline, Waterfall |
| **Marketing Director** | Find top-performing countries and channels | Choropleth map, Sankey flow |
| **Product Manager** | Understand which categories drive orders | Treemap, Category bar |
| **CRM / Retention Team** | Identify loyal vs. at-risk customer segments | RFM analysis, Loyalty bubble |
| **Operations** | Plan staffing by peak hours and days | 24-hr clock, Day-of-week heatmap |

---

## 2. Problem Statement

### 2.1 Core Question

> **How can a retail business efficiently identify revenue trends, top-performing segments, and customer loyalty patterns in online sales data — without relying on manual SQL queries or spreadsheet exports?**

More specifically, the dashboard will answer:
- How is revenue evolving month by month, and are there seasonal peaks?
- Which product categories and countries generate the most revenue?
- Which payment methods and sales channels dominate transactions?
- How do customers segment by loyalty (repeat buyers vs. one-time buyers)?
- At what hours and days of the week do sales concentrate?

### 2.2 Why This Matters Statistically

In retail analytics, descriptive statistics alone are insufficient. The **distribution of revenue is highly skewed** — a small proportion of customers typically account for a disproportionate share of revenue (the Pareto principle, often cited as the 80/20 rule in CRM literature). Visualizing this skew requires more than a mean and standard deviation; it requires interactive segment exploration. Similarly, **geographic concentration** of sales is not visible in aggregate numbers but becomes immediately clear in a choropleth map.

---

## 3. Dataset Description

### 3.1 Dataset Source

For this project, I use the **Online Sales Dataset** — a synthetic but realistic e-commerce transactions dataset containing invoice-level order records. The dataset is structured to reflect real-world online retail operations and includes dimensions across time, geography, product category, payment behavior, and sales channel.

**File:** `online_sales_dataset.csv`  
**Format:** CSV, comma-delimited  
**Origin:** Structured e-commerce transaction records for academic visualization projects

### 3.2 Why This Dataset?

This dataset was selected for three key reasons:

1. **Multi-dimensional richness** — It contains temporal (date, hour), geographic (country), behavioral (payment method, sales channel), and product (category) dimensions. This allows us to build a wide variety of chart types (time series, maps, sunburst, heatmaps) from a single source — essential for a visualization project.

2. **Real-world relevance** — The columns mirror what companies like Amazon, Shopify, and Walmart collect at every transaction. The insights from this dashboard are directly transferable to industry practice.

3. **Customer-level analysis** — The presence of a `CustomerID` column allows customer segmentation (RFM analysis), which is one of the most impactful analytics tasks in e-commerce as documented by Ngai et al. (2009).

### 3.3 Dataset Structure

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = '#f8f9fa'
plt.rcParams['font.family'] = 'sans-serif'

# Load dataset
df = pd.read_csv('/Users/neenubonny/Downloads/online_sales_dataset.csv')

print('=' * 50)
print('DATASET OVERVIEW')
print('=' * 50)
print(f'Total records    : {len(df):,}')
print(f'Total columns    : {len(df.columns)}')
print(f'Memory usage     : {df.memory_usage(deep=True).sum() / 1024:.1f} KB')
print()
print('Columns:')
for col in df.columns:
    print(f'  {col:<25} {df[col].dtype}')

In [ ]:
# Show first few rows
df.head()

### 3.4 Preprocessing & Derived Columns

In [ ]:
# Parse dates
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Standardize PaymentMethod typo
df['PaymentMethod'] = df['PaymentMethod'].replace({'Paypall': 'PayPal', 'paypall': 'PayPal'})

# Derive key columns
df['Revenue']    = df['Quantity'] * df['UnitPrice'] * (1 - df['Discount'].fillna(0))
df['Profit']     = df['Revenue'] - df['ShippingCost'].fillna(0)
df['Month']      = df['InvoiceDate'].dt.to_period('M').astype(str)
df['Year']       = df['InvoiceDate'].dt.year
df['Hour']       = df['InvoiceDate'].dt.hour
df['DayOfWeek']  = df['InvoiceDate'].dt.day_name()
df['IsReturned'] = df['ReturnStatus'].str.lower().str.contains('return', na=False)
df['IsOnline']   = df['SalesChannel'].str.lower().str.contains('online', na=False)

print('Date range   :', df['InvoiceDate'].min().date(), 'to', df['InvoiceDate'].max().date())
print('Unique customers :', df['CustomerID'].nunique():,)
print('Unique countries :', df['Country'].nunique())
print('Derived columns added: Revenue, Profit, Month, Year, Hour, DayOfWeek, IsReturned, IsOnline')

---
## 4. Exploratory Data Analysis (EDA)

### 4.1 Missing Values

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

if missing_df.empty:
    print('No missing values found. Dataset is complete.')
else:
    print(missing_df)

### 4.2 Key Descriptive Statistics

In [ ]:
total_revenue   = df['Revenue'].sum()
total_orders    = len(df)
unique_customers = df['CustomerID'].nunique()
avg_order_value = df['Revenue'].mean()
return_rate     = df['IsReturned'].mean() * 100
online_pct      = df['IsOnline'].mean() * 100

print('=' * 45)
print('KEY BUSINESS METRICS')
print('=' * 45)
print(f'Total Revenue        : ${total_revenue:>12,.2f}')
print(f'Total Orders         : {total_orders:>12,}')
print(f'Unique Customers     : {unique_customers:>12,}')
print(f'Avg Order Value      : ${avg_order_value:>12,.2f}')
print(f'Return Rate          : {return_rate:>11.1f}%')
print(f'Online Channel Share : {online_pct:>11.1f}%')
print()
print('Revenue Distribution:')
print(df['Revenue'].describe().apply(lambda x: f'${x:,.2f}'))

### 4.3 Revenue Over Time

In [ ]:
monthly = df.groupby('Month')['Revenue'].sum().reset_index()
monthly = monthly.sort_values('Month')

fig, ax = plt.subplots(figsize=(14, 4))
ax.fill_between(range(len(monthly)), monthly['Revenue'], alpha=0.3, color='#3b82f6')
ax.plot(range(len(monthly)), monthly['Revenue'], color='#3b82f6', linewidth=2.5, marker='o', markersize=5)
ax.set_xticks(range(len(monthly)))
ax.set_xticklabels(monthly['Month'], rotation=45, ha='right', fontsize=8)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
ax.set_title('Monthly Revenue Trend', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Month')
ax.set_ylabel('Revenue')
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

peak_month = monthly.loc[monthly['Revenue'].idxmax(), 'Month']
print(f'Peak month: {peak_month} — ${monthly["Revenue"].max():,.0f}')

### 4.4 Revenue by Product Category

In [ ]:
cat_rev = df.groupby('Category')['Revenue'].sum().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(cat_rev.index, cat_rev.values, color='#6366f1', edgecolor='white')
for bar, val in zip(bars, cat_rev.values):
    ax.text(val + cat_rev.max() * 0.01, bar.get_y() + bar.get_height()/2,
            f'${val/1e3:.0f}K', va='center', fontsize=10)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
ax.set_title('Revenue by Product Category', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Total Revenue')
ax.grid(axis='x', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

### 4.5 Top 10 Countries by Revenue

In [ ]:
country_rev = df.groupby('Country')['Revenue'].sum().sort_values(ascending=False).head(10)

fig, ax = plt.subplots(figsize=(11, 4))
colors = ['#3b82f6' if i == 0 else '#93c5fd' for i in range(len(country_rev))]
bars = ax.bar(country_rev.index, country_rev.values, color=colors, edgecolor='white')
for bar, val in zip(bars, country_rev.values):
    ax.text(bar.get_x() + bar.get_width()/2, val + country_rev.max()*0.01,
            f'${val/1e3:.0f}K', ha='center', fontsize=9)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
ax.set_title('Top 10 Countries by Revenue', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Country')
ax.set_ylabel('Revenue')
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

### 4.6 Sales Channel & Payment Method Breakdown

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Sales Channel
ch = df.groupby('SalesChannel')['Revenue'].sum()
axes[0].pie(ch.values, labels=ch.index, autopct='%1.1f%%',
            colors=['#3b82f6','#10b981'], startangle=90,
            wedgeprops=dict(edgecolor='white', linewidth=2))
axes[0].set_title('Revenue by Sales Channel', fontweight='bold')

# Payment Method
pm = df.groupby('PaymentMethod')['Revenue'].sum().sort_values(ascending=True)
axes[1].barh(pm.index, pm.values, color='#f59e0b', edgecolor='white')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
axes[1].set_title('Revenue by Payment Method', fontweight='bold')
axes[1].grid(axis='x', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()

### 4.7 Hourly Sales Distribution (24-Hour Pattern)

In [ ]:
hourly = df.groupby('Hour')['Revenue'].sum()

fig, ax = plt.subplots(figsize=(13, 4))
ax.bar(hourly.index, hourly.values, color='#8b5cf6', edgecolor='white')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
ax.set_title('Revenue by Hour of Day', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Hour (0 = midnight, 12 = noon)')
ax.set_ylabel('Revenue')
ax.set_xticks(range(24))
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

peak_hr = hourly.idxmax()
print(f'Peak sales hour: {peak_hr}:00 — ${hourly.max():,.0f}')

### 4.8 Revenue Distribution — Is It Skewed?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
axes[0].hist(df['Revenue'].clip(upper=df['Revenue'].quantile(0.99)),
             bins=60, color='#3b82f6', edgecolor='white', alpha=0.8)
axes[0].set_title('Revenue Distribution (clipped at 99th pct)', fontweight='bold')
axes[0].set_xlabel('Revenue per Order ($)')
axes[0].set_ylabel('Frequency')
axes[0].grid(axis='y', linestyle='--', alpha=0.4)

# Box plot by category
cats = df['Category'].unique()
data = [df[df['Category'] == c]['Revenue'].clip(upper=df['Revenue'].quantile(0.95)).values for c in cats]
bp = axes[1].boxplot(data, labels=cats, patch_artist=True)
colors_box = ['#3b82f6','#10b981','#f59e0b','#ef4444','#8b5cf6']
for patch, color in zip(bp['boxes'], colors_box[:len(cats)]):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].set_title('Revenue Spread by Category', fontweight='bold')
axes[1].set_ylabel('Revenue ($)')
axes[1].tick_params(axis='x', rotation=20)
axes[1].grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()

skewness = df['Revenue'].skew()
print(f'Revenue skewness: {skewness:.2f} — {"Right-skewed (few high-value orders dominate)" if skewness > 1 else "Approximately normal"}')

### 4.9 Customer Purchase Frequency

In [ ]:
cust_freq = df.groupby('CustomerID').size()
repeat_buyers = (cust_freq > 1).sum()
one_time = (cust_freq == 1).sum()

print(f'One-time buyers : {one_time:,} ({one_time/len(cust_freq)*100:.1f}%)')
print(f'Repeat buyers   : {repeat_buyers:,} ({repeat_buyers/len(cust_freq)*100:.1f}%)')
print(f'Max orders by 1 customer: {cust_freq.max()}')
print(f'Avg orders per customer : {cust_freq.mean():.2f}')

fig, ax = plt.subplots(figsize=(9, 4))
freq_dist = cust_freq.value_counts().sort_index().head(15)
ax.bar(freq_dist.index, freq_dist.values, color='#10b981', edgecolor='white')
ax.set_title('Customer Purchase Frequency Distribution', fontsize=13, fontweight='bold')
ax.set_xlabel('Number of Orders per Customer')
ax.set_ylabel('Number of Customers')
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

---
## 5. Dashboard Prototype

The full interactive dashboard has been built in **Streamlit** and is running locally at `http://localhost:8508`. Below is a summary of the prototype visualizations already implemented:

| # | Chart | Type | Interaction |
|---|---|---|---|
| 1 | Revenue & Orders Over Time | Line chart | Hover tooltips, filter by year |
| 2 | Revenue by Category | Bar chart | Sidebar category filter |
| 3 | Geographic Revenue Map | Choropleth | Hover by country |
| 4 | Payment × Channel | Sunburst | Click to drill down |
| 5 | Sales Journey Flow | Sankey diagram | Hover on flows |
| 6 | Revenue Waterfall | Waterfall chart | Gross → Discounts → Shipping → Net |
| 7 | 24-Hour Sales Clock | Polar bar chart | Hour-level hover |
| 8 | Day × Category Heatmap | Heatmap | Hover tooltips |
| 9 | Customer Loyalty Bubble | Bubble chart | Hover with order details |
| 10 | Parallel Coordinates | Parcoords | Brush to filter axes |
| 11 | AI Chatbot | Natural language | Type any business question |

**Sidebar interactions implemented:**
- Country multi-select filter
- Product category multi-select filter
- Sales channel toggle (Online / In-store / Both)
- Year range slider

All charts update dynamically when sidebar filters change.

---

## 6. What's Next (Milestone 2 & Final)

- Add formal **RFM customer segmentation** with Champions / Loyal / At-Risk labels
- Deploy to **Streamlit Cloud** for a public shareable URL
- Add **cohort retention analysis** (how many customers return month over month)
- Polish visual design and add a narrative introduction panel

---

## 7. References

1. Sarikaya, A., Correll, M., Bartram, L., Tory, M., & Fisher, D. (2019). *What do we talk about when we talk about dashboards?* IEEE Transactions on Visualization and Computer Graphics, 25(1), 682–692. https://doi.org/10.1109/TVCG.2018.2864903

2. Heer, J., & Shneiderman, B. (2012). *Interactive dynamics for visual analysis.* Queue, 10(2), 30–55. https://doi.org/10.1145/2133416.2146416

3. Ngai, E.W.T., Xiu, L., & Chau, D.C.K. (2009). *Application of data mining techniques in customer relationship management: A literature review and classification.* Expert Systems with Applications, 36(2), 2592–2602. https://doi.org/10.1016/j.eswa.2007.10.017

4. Chen, I.J., & Popovich, K. (2003). *Understanding customer relationship management (CRM): People, process and technology.* Business Process Management Journal, 9(5), 672–688. https://doi.org/10.1108/14637150310496758

5. Statista. (2024). *E-commerce worldwide — Statistics & Facts.* https://www.statista.com/topics/871/online-shopping/